# Import prepared datasets as Delta tables

Runs **inside Databricks**, not locally — it uses `spark` and `dbutils`,
which only exist in a Databricks notebook environment. Import this file
into your workspace (Workspace → Import → this `.ipynb`) rather than
running it from a local Jupyter kernel.

## Before running: upload the 4 files

This notebook creates a Unity Catalog **Volume** and expects the following
files (from the local `analysis/` folder) to be uploaded into it first:

- `short_stay_listing_features.csv`
- `short_stay_review_text.csv`
- `short_stay_listing_proximity.csv`
- `poi/singapore_poi.csv` (upload as `singapore_poi.csv`, flattened — no subfolder)

**Steps:**
1. Run the first code cell below once — it prints the exact Volume path
   (e.g. `/Volumes/<catalog>/booking_system/raw_files`) and creates the
   schema/volume if they don't exist yet.
2. In the workspace sidebar: **Catalog** → navigate to that
   catalog/schema/volume → **Upload to this volume** → drag in the 4 files
   (renaming `poi/singapore_poi.csv` to just `singapore_poi.csv` on upload).
3. Come back and run the rest of the notebook.

Adjust the widgets below first if you want a different catalog/schema/volume
name than the defaults.

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog (blank = current default)")
dbutils.widgets.text("schema", "booking_system", "Schema name")
dbutils.widgets.text("volume", "raw_files", "Volume name (for uploaded CSVs)")

catalog = dbutils.widgets.get("catalog") or spark.sql("SELECT current_catalog()").first()[0]
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
print(f"Catalog.schema for the Delta tables: {catalog}.{schema}")
print(f"Upload the 4 CSVs to this Volume path: {volume_path}")

## Load helpers

Every id-like column (`id`, `host_id`, `host_profile_id`, `listing_id`,
`osm_id`) is explicitly cast to a string after load. These are 19-digit
numbers in this dataset — Spark's `LongType` (64-bit) actually holds them
exactly, no precision loss like a language limited to 64-bit *floats*
would have (that bit the Node.js backend earlier, and a float64 upcast on
a nullable pandas int column bit `host_profile_id` in the CSV itself,
before it was corrected at the source). Casting to string here isn't
fixing a precision problem — Spark doesn't have one — it's just policy:
identifiers aren't used arithmetically, so they don't need to be numeric.

In [ ]:
from pyspark.sql.functions import col, split, when, trim

def load_csv(filename):
    # These CSVs have text cells that contain line breaks (e.g. host_about)
    # and quote characters written as "" (standard CSV). Spark's defaults
    # assume one row per physical line and a backslash escape, so without
    # these three options one listing gets split into several broken rows.
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)   # a quoted cell may span several lines
        .option("quote", '"')
        .option("escape", '"')       # "" inside a quoted cell means one literal quote
        .csv(f"{volume_path}/{filename}")
    )

def cast_ids(df, *id_columns):
    for c in id_columns:
        if c in df.columns:
            df = df.withColumn(c, col(c).cast("string"))
    return df

def cast_flags(df, *flag_columns):
    # source CSVs encode booleans as "t"/"f" text
    for c in flag_columns:
        if c in df.columns:
            df = df.withColumn(c, when(col(c) == "t", True).when(col(c) == "f", False))
    return df

## Listing features → `listing_features`

In [ ]:
listing_features = load_csv("short_stay_listing_features.csv")
listing_features = cast_ids(listing_features, "id", "host_id", "host_profile_id")
listing_features = cast_flags(
    listing_features,
    "has_availability", "host_is_superhost", "host_identity_verified", "host_has_profile_pic",
)
listing_features = listing_features.withColumn(
    "amenities_array", split(trim(col("amenities")), ",\\s*")
)

# overwriteSchema: if the table already exists from an earlier run with different
# column types, replace its schema too (plain "overwrite" only replaces the rows)
listing_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema}.listing_features"
)
print(f"listing_features: {listing_features.count():,} rows")

## Review text → `review_text`

In [ ]:
review_text = load_csv("short_stay_review_text.csv")
review_text = cast_ids(review_text, "listing_id")

review_text.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema}.review_text"
)
print(f"review_text: {review_text.count():,} rows")

## Nearby-places proximity → `listing_proximity`

In [ ]:
listing_proximity = load_csv("short_stay_listing_proximity.csv")
listing_proximity = cast_ids(listing_proximity, "listing_id")

listing_proximity.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{catalog}.{schema}.listing_proximity"
)
print(f"listing_proximity: {listing_proximity.count():,} rows")

## Points of interest → `poi`

In [ ]:
poi = load_csv("singapore_poi.csv")
poi = cast_ids(poi, "osm_id")

poi.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.poi")
print(f"poi: {poi.count():,} rows")

## Verify

In [ ]:
expected = {"listing_features": 1483, "review_text": 1483, "listing_proximity": 1483, "poi": 2000}

for t, want in expected.items():
    n = spark.table(f"{catalog}.{schema}.{t}").count()
    status = "OK" if n == want else "<-- MISMATCH, check the CSV upload / reader options"
    print(f"{catalog}.{schema}.{t}: {n:,} rows (expected {want:,})  {status}")

display(spark.table(f"{catalog}.{schema}.listing_features").limit(5))

## Next

These four Delta tables are the foundation for the next steps: generating
and storing text embeddings (`listing_features.description` /
`review_text.review_text`), a Vector Search index over them, and MLflow
for tracking whatever ranking model comes after that.